In [138]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [139]:
CSV_PATH="raw_car_dataset.csv"
df=pd.read_csv(CSV_PATH)

In [140]:
print(df.info)


<bound method DataFrame.info of       Price  Odometer_km  Doors  Accidents Location  Year
0    $1,500     137830.0    4.0          1     City  1998
1    4171.0          NaN    4.0          0    Rural  2016
2    $5,331     107302.0    4.0          0   Suburb  2014
3    1500.0     141838.0    4.0          1   Suburb  1999
4    1500.0          NaN    3.0          0     City  2022
..      ...          ...    ...        ...      ...   ...
140  1500.0     223193.0    3.0          0     City  2003
141  1500.0     124567.0    NaN          2   Suburb  2002
142  1500.0     203153.0    4.0          0   Suburb  2004
143  1500.0     180030.0    4.0          1     City  2009
144  $1,500     211171.0    4.0          0       ??  2003

[145 rows x 6 columns]>


In [141]:
print(df.shape)

(145, 6)


In [142]:
print(df.head())


    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022


In [143]:
print(df.isnull(). sum())

Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


In [144]:
print(df["Location"].value_counts(dropna=False))

Location
City      59
Suburb    45
Rural     21
Subrb      8
??         7
NaN        5
Name: count, dtype: int64


In [145]:
df["Price"]=df["Price"].astype(str).str.replace(r"[/$,]","", regex=True).astype(float)

In [146]:
print(df.head())

    Price  Odometer_km  Doors  Accidents Location  Year
0  1500.0     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  5331.0     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022


In [147]:
df["Location"]=df["Location"].replace({"Subrb":"Suburb","??" : pd.NA})

In [148]:
print(df.head(10))

    Price  Odometer_km  Doors  Accidents Location  Year
0  1500.0     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  5331.0     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022
5  1500.0     211171.0    4.0          0     <NA>  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  1500.0     105068.0    5.0          1     City  2002
8  2291.0      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


In [149]:
#filling the null
df["Odometer_km"]=df["Odometer_km"].fillna(df["Odometer_km"].median())
df["Doors"]=df["Doors"].fillna(df["Doors"].mode()[0])
df["Location"]=df["Location"].fillna(df["Location"].mode()[0])
print(df.isnull().sum())


Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


In [150]:
#removing duplicates
df=df.drop_duplicates()
print(df.shape)
print(df["Price"].skew())

(140, 6)
7.73557540733881


In [151]:
#outlier
def iqr_func(series,k=1.5):
   q1,q3=series.quantile([0.25,0.75])
   iq=q3-q1
   lower= q1=q1-k*iq
   higher=q3=q3+k*iq
   return lower,higher
low_Odometer,high_Odometer=iqr_func(df["Odometer_km"])
low_price,high_price=iqr_func(df["Price"])
print(low_Odometer,high_Odometer)
print(df["Price"].skew())
                              
    




-6642.25 271987.75
7.73557540733881


In [152]:
## limiting all price and size_sqft to lower and upper calculated 
df["Odometer_km"]=df["Odometer_km"].clip(lower=low_Odometer, upper=high_Odometer)
df["Price"]=df["Price"].clip(lower=low_price, upper=high_price)

In [153]:
print(df.head())
print(df["Price"].skew())


    Price  Odometer_km  Doors  Accidents Location  Year
0  1500.0     137830.0    4.0          1     City  1998
1  4171.0     128548.0    4.0          0    Rural  2016
2  5331.0     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0     128548.0    3.0          0     City  2022
1.2904627148773138


In [154]:
# creating one hot encodin
df=pd.get_dummies(df,columns=["Location"],drop_first=True , dtype=int )
print(df.head(10))


    Price  Odometer_km  Doors  Accidents  Year  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998               0   
1  4171.0     128548.0    4.0          0  2016               1   
2  5331.0     107302.0    4.0          0  2014               0   
3  1500.0     141838.0    4.0          1  1999               0   
4  1500.0     128548.0    3.0          0  2022               0   
5  1500.0     211171.0    4.0          0  2003               0   
6  1500.0     222235.0    4.0          2  2004               1   
7  1500.0     105068.0    5.0          1  2002               0   
8  2291.0      90015.0    4.0          0  2007               1   
9  1500.0     125976.0    2.0          0  2002               0   

   Location_Suburb  
0                0  
1                0  
2                1  
3                1  
4                0  
5                0  
6                0  
7                0  
8                0  
9                0  


In [155]:
#feature engineering
CURRENT_YEAR=2026
df["Car_age"]=CURRENT_YEAR - df["Year"]
df["Km_Per_Year"] = df["Odometer_km"] / df["Car_age"].replace(0,np.nan)
df["Age_x_Odometer"] = df["Car_age"] * df["Odometer_km"]
print(df.head(10))


    Price  Odometer_km  Doors  Accidents  Year  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998               0   
1  4171.0     128548.0    4.0          0  2016               1   
2  5331.0     107302.0    4.0          0  2014               0   
3  1500.0     141838.0    4.0          1  1999               0   
4  1500.0     128548.0    3.0          0  2022               0   
5  1500.0     211171.0    4.0          0  2003               0   
6  1500.0     222235.0    4.0          2  2004               1   
7  1500.0     105068.0    5.0          1  2002               0   
8  2291.0      90015.0    4.0          0  2007               1   
9  1500.0     125976.0    2.0          0  2002               0   

   Location_Suburb  Car_age   Km_Per_Year  Age_x_Odometer  
0                0       28   4922.500000       3859240.0  
1                0       10  12854.800000       1285480.0  
2                1       12   8941.833333       1287624.0  
3                1       27   525

In [156]:
df["Logprice"]=np.log1p(df["Price"])

In [157]:
print(df.head())

    Price  Odometer_km  Doors  Accidents  Year  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998               0   
1  4171.0     128548.0    4.0          0  2016               1   
2  5331.0     107302.0    4.0          0  2014               0   
3  1500.0     141838.0    4.0          1  1999               0   
4  1500.0     128548.0    3.0          0  2022               0   

   Location_Suburb  Car_age   Km_Per_Year  Age_x_Odometer  Logprice  
0                0       28   4922.500000       3859240.0  7.313887  
1                0       10  12854.800000       1285480.0  8.336151  
2                1       12   8941.833333       1287624.0  8.581482  
3                1       27   5253.259259       3829626.0  7.313887  
4                0        4  32137.000000        514192.0  7.313887  


In [165]:
#Feature scaling (X only; keep targets & dummies unscaled)
ignore_scale={"Price","Logprice"}
col_scaling=df.select_dtypes(include=["int64","float64"]).columns.tolist()
exclude = [c for c in df.columns if c.startswith("Location_")]
num_features_to_scale = [c for c in col_scaling if c not in ignore_scale and c not in exclude]
scaler = StandardScaler()
df[num_features_to_scale] = scaler.fit_transform(df[num_features_to_scale])
print(df.head())




    Price  Odometer_km     Doors  Accidents      Year  Location_Rural  \
0  1500.0     0.128390  0.254091   0.316968 -1.686714               0   
1  4171.0    -0.044709  0.254091  -0.820867  0.794617               1   
2  5331.0    -0.440923  0.254091  -0.820867  0.518913               0   
3  1500.0     0.203135  0.254091   0.316968 -1.548862               0   
4  1500.0    -0.044709 -0.931668  -0.820867  1.621727               0   

   Location_Suburb   Car_age  Km_Per_Year  Age_x_Odometer  Logprice  
0                0  1.686714    -0.615631        1.418432  7.313887  
1                0 -0.794617     0.070446       -0.584927  8.336151  
2                1 -0.518913    -0.267993       -0.583258  8.581482  
3                1  1.548862    -0.587024        1.395381  7.313887  
4                0 -1.621727     1.738196       -1.185281  7.313887  


In [169]:
#saving
OUT_PATH="car_dataset_cleaned.csv"
df.to_csv(OUT_PATH ,index=False)